In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
import xgboost as xgb
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score

In [ ]:
df=pd.read_csv('train_data.csv')

In [ ]:
df.head()

In [ ]:
# Handle NaN values with central tendencies
for col in df.columns:
    if df[col].isna().sum() > 0:
        if df[col].dtype == 'object':
            df[col].fillna(df[col].mode()[0], inplace=True)
        else:
            df[col].fillna(df[col].median(), inplace=True)

In [ ]:
# Define target column
target_col = "bad_flag"

# Split data based on target values
num_1 = int(len(df[df[target_col] == 1]) * 0.45)
num_0 = int(len(df[df[target_col] == 0]) * 0.55)

data_1 = df[df[target_col] == 1].sample(n=num_1, random_state=42)
data_0 = df[df[target_col] == 0].sample(n=num_0, random_state=42)

In [ ]:
# Create training data
train_data = pd.concat([data_1, data_0]).sample(frac=1, random_state=42).reset_index(drop=True)
X_train = train_data.drop(columns=[target_col])
y_train = train_data[target_col]

In [ ]:
# The remaining data will be used as test data
remaining_data = df.drop(train_data.index)
X_test = remaining_data.drop(columns=[target_col])
y_test = remaining_data[target_col]

In [ ]:
# Scale data
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [ ]:
# Calculate scale_pos_weight (ratio of class 0 to class 1)
num_0 = len(df[df[target_col] == 0])
num_1 = len(df[df[target_col] == 1])
scale_pos_weight = num_0 / num_1  # Penalizes misclassification of fraud cases

In [ ]:
# Train XGBoost with class weighting
model = xgb.XGBClassifier(
    use_label_encoder=False,
    eval_metric='logloss',
    scale_pos_weight=scale_pos_weight  # Balances class weights
)
model.fit(X_train, y_train)


In [ ]:
# Get probability predictions
y_prob = model.predict_proba(X_test)[:, 1]

# Adjust threshold for better fraud detection
threshold = 0.35  # Lower threshold to flag more fraud cases
y_pred = (y_prob > threshold).astype(int)

In [ ]:
# Evaluate model
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)

# Print evaluation metrics
print(f'Accuracy: {accuracy:.4f}')
print(f'F1 Score: {f1:.4f}')
print(f'Precision: {precision:.4f}')
print(f'Recall: {recall:.4f}')
print(f'Confusion Matrix:\n{conf_matrix}')
print(f'ROC AUC: {roc_auc:.4f}')